In [1]:
# pip install Unidecode

In [1]:
# ------------------  SEDS cleaner  ------------------
import pandas as pd
from pathlib import Path

RAW_DIR   = Path(r"D:\CIM Warwick\Dissertation\Datasets\All Datasets")          
CLEAN_DIR = RAW_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

SEDS_FILES = {
    "Complete_SEDS.csv"     : "complete",
    "use_US.csv"            : "consumption",
    "pr_US.csv"             : "prices",
    "co2_US.csv"            : "co2",
    "energy_indicators.csv" : "indicators",
    "prod_data.xlsx"     : "production",
}

# ------------ helper utils (put near the top of your script) ----------------
def _normalize_cols(df):
    """Return columns as lower-case strings for easy matching."""
    return [str(c).strip().lower() for c in df.columns]

def _detect_format(cols):
    """Return 'long' if a column literally named 'year' exists, else 'wide'."""
    return "long" if "year" in cols else "wide"


def tidy_seds(path: Path, tag: str) -> pd.DataFrame:
    # --- 1. read -------------------------------------------------------------
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        df = pd.read_excel(path, sheet_name=0)

    cols_norm = _normalize_cols(df)                     #### NEW
    fmt = _detect_format(cols_norm)                     #### NEW

    # --- 2. locate canonical columns ----------------------------------------
    # use cols_norm to find positions, then map back to original names
    colmap = {norm: orig for norm, orig in zip(cols_norm, df.columns)}

    state_col = colmap.get("state") or colmap.get("statecode")
    msn_col   = colmap.get("msn")

    if not msn_col:
        raise ValueError(f"MSN column missing in {path.name}")

    if not state_col:
        # Assume file is a national aggregate (e.g., Prod_dataset)
        df["state"] = "US"
        state_col = "state"

    # --- 3. tidy long vs. melt ---------------------------------------------
    if fmt == "long":
        year_col = colmap["year"]
        data_col = next(c for c in df.columns if str(c).lower() == "data")

        tidy = (
            df[[state_col, msn_col, year_col, data_col]]
            .rename(columns={
                state_col: "state",
                msn_col  : "msn",
                year_col : "year",
                data_col : "value",
            })
        )
    else:  # wide
        year_cols = [c for c in df.columns if str(c).isdigit()]
        tidy = (
            df.melt(
                id_vars=[state_col, msn_col],
                value_vars=year_cols,
                var_name="year",
                value_name="value",
            )
            .rename(columns={state_col: "state", msn_col: "msn"})
        )
        tidy["year"] = tidy["year"].astype(int)

    # --- 4. post-processing --------------------------------------------------
    tidy["value"] = (
        tidy["value"]
            .replace(["Not Available", "NA", ".", ""], pd.NA)
            .astype("float64")
    )
    tidy["dataset"] = tag
    return tidy[["state", "msn", "year", "value", "dataset"]]


# ------------------  run on every file --------------------------------------
frames = []
for fname, tag in SEDS_FILES.items():
    fpath = RAW_DIR / fname
    if not fpath.exists():
        print(f"⚠️  {fname} not found – skipping")
        continue
    df_clean = tidy_seds(fpath, tag)
    out = CLEAN_DIR / f"{tag}_tidy.parquet"
    df_clean.to_parquet(out, index=False)
    frames.append(df_clean)
    print(f"✓ {fname} → {out} ({len(df_clean):,} rows)")

# Optional: one big parquet
if frames:
    big = pd.concat(frames, ignore_index=True)
    big.to_parquet(CLEAN_DIR / "seds_all.parquet", index=False)
    print(f"✨ combined tidy file saved as  {CLEAN_DIR/'seds_all.parquet'}")


✓ Complete_SEDS.csv → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\complete_tidy.parquet (2,366,854 rows)
✓ use_US.csv → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\consumption_tidy.parquet (35,392 rows)
✓ pr_US.csv → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\prices_tidy.parquet (19,602 rows)
✓ co2_US.csv → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\co2_tidy.parquet (2,176 rows)
✓ energy_indicators.csv → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\indicators_tidy.parquet (242,944 rows)
✓ prod_data.xlsx → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\production_tidy.parquet (113,856 rows)
✨ combined tidy file saved as  D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\seds_all.parquet


In [6]:
# ------------ ultra-robust UKPN cleaner (essentials only) -------------
import pandas as pd, re
from pathlib import Path

# Accept whichever of these exists (parquet preferred)
CANDIDATES = [
    RAW_DIR / "ukpn-data-centre-demand-profiles.parquet",
    RAW_DIR / "ukpn-data-centre-demand-profiles.csv",
    RAW_DIR / "ukpn-data-centre-utilisation.parquet",      # fallback
    RAW_DIR / "ukpn-data-centre-utilisation.csv",
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("No UKPN demand-profile file found")

file_path = first_existing(CANDIDATES)
print(f"→ Loading {file_path.name}")

df = (
    pd.read_parquet(file_path) if file_path.suffix == ".parquet"
    else pd.read_csv(file_path, low_memory=False)
)

print("Columns detected:\n", df.columns.tolist()[:20], "…")  # show first 20

# ---------------------------------------------------------------------
# 1. Find the datetime & utilisation columns *by pattern*  -------------
# ---------------------------------------------------------------------
datetime_col = next(
    (c for c in df.columns if re.search(r"datetime", str(c), re.I)),
    None
)
util_col = next(
    # look for 'utilisation' and either 'half' OR '%' to avoid the max-only column
    (c for c in df.columns if re.search(r"utilisation", str(c), re.I)
                            and (re.search(r"half|hour", str(c), re.I) or "%" in str(c))),
    None
)
site_col = next(
    (c for c in df.columns if re.search(r"anonymised", str(c), re.I) or re.search(r"name", str(c), re.I)),
    None
)

dctype_col = next(
    (c for c in df.columns if re.search(r"derived", str(c), re.I) or re.search(r"type", str(c), re.I)),
    None
)

if not all([datetime_col, util_col, site_col, dctype_col]):
    raise ValueError(
        "Couldn’t find required columns — please copy the column list above "
        "into chat so we can update the matcher."
    )

print(f"✔ Using columns → datetime: '{datetime_col}', utilisation: '{util_col}', site_id: '{site_col}', dctype: '{dctype_col}'")

# ---------------------------------------------------------------------
# 2. Minimal clean  ----------------------------------------------------
# ---------------------------------------------------------------------
tidy = pd.DataFrame({
    "site_id"        : df[site_col].astype(str),
    "datetime_local" : pd.to_datetime(df[datetime_col], errors="coerce", utc=True),
    "utilisation"    : pd.to_numeric(df[util_col], errors="coerce") / 100.0  # 0–1
}).dropna()

out_file = CLEAN_DIR / "ukpn_timeseries_tidy.parquet"
tidy.to_parquet(out_file, index=False)
print(f"✨ Cleaned half-hourly file saved → {out_file}  ({len(tidy):,} rows)")


→ Loading ukpn-data-centre-demand-profiles.parquet
Columns detected:
 ['anonymised_name', 'data_centre_type_derived', 'cleansed_voltage_level', 'datetime_local', 'rounded_data_centre_half_hourly_utilisation'] …
✔ Using columns → datetime: 'datetime_local', utilisation: 'rounded_data_centre_half_hourly_utilisation', site_id: 'anonymised_name', dctype: 'data_centre_type_derived'
✨ Cleaned half-hourly file saved → D:\CIM Warwick\Dissertation\Datasets\All Datasets\clean\ukpn_timeseries_tidy.parquet  (1,580,697 rows)


In [5]:
# ───────── Work-station power traces • FINAL CLEANER ─────────

import unidecode

RAW  = Path(RAW_DIR / "1mayo - agosto 2021.csv")
OUT  = Path(CLEAN_DIR /"workstation_2021_may_aug_tidy.parquet")

HEADER_MAP = {
    "mac"                    : "mac",
    "weekday"                : "weekday",
    "fecha_servidor"         : "datetime",          # rename to datetime
    "voltaje"                : "voltage_v",
    "corriente"              : "current_a",
    "potencia"               : "power_w",
    "frecuencia"             : "frequency_hz",
    "energia"                : "active_energy_kwh",
    "fp"                     : "power_factor",
    "esp32_temp"             : "esp32_temp_c",
    "workstation_cpu"        : "cpu_util_pct",
    "workstation_cpu_power"  : "cpu_power_pct",
    "workstation_cpu_temp"   : "cpu_temp_c",
    "workstation_gpu"        : "gpu_util_pct",
    "workstation_gpu_power"  : "gpu_power_pct",
    "workstation_gpu_temp"   : "gpu_temp_c",
    "workstation_ram"        : "ram_util_pct",
    "workstation_ram_power"  : "ram_power_pct",
}

def tidy_may_aug(csv_path: Path) -> pd.DataFrame:
    # fast C-engine; BOM handled
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    # normalise column names then map to English
    df.columns = [unidecode.unidecode(c).lower().strip('" ') for c in df.columns]
    df = df.rename(columns={k: HEADER_MAP.get(k, k) for k in df.columns})

    # drop duplicate ESP timestamp if present
    df = df.drop(columns=["fecha_esp32"], errors="ignore")

    # parse datetime (day-first) & sort
    df["datetime"] = pd.to_datetime(
        df["datetime"], dayfirst=True, utc=True, errors="coerce"
    )
    df = df.dropna(subset=["datetime"]).sort_values("datetime")

    # cast all metric columns to numeric
    for col in df.columns.difference(["datetime", "mac", "weekday"]):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

print("→ Cleaning 1mayo – agosto 2021.csv …")
may_aug = tidy_may_aug(RAW)
print(f"Parsed datetimes: {may_aug['datetime'].notna().sum():,} rows")

may_aug.to_parquet(OUT, index=False)
print(f"✓ wrote {OUT.name}  –  {len(may_aug):,} rows  "
      f"|  {round(OUT.stat().st_size / 1024**2, 1)} MB")




→ Cleaning 1mayo – agosto 2021.csv …
Parsed datetimes: 2,889,533 rows
✓ wrote workstation_2021_may_aug_tidy.parquet  –  2,889,533 rows  |  23.8 MB


In [6]:
import csv
# ───────── Work-station power traces • AUG–DEC 2021 ─────────
RAW = Path(RAW_DIR / "2agosto -dic 2021.csv")
OUT = Path(CLEAN_DIR /"workstation_2021_aug_dec_tidy.parquet")

def read_fast_then_fallback(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")                      # fast C-engine
    except pd.errors.ParserError:
        # fallback: tolerant Python engine, skipping malformed rows
        return pd.read_csv(path,
                           encoding="utf-8-sig",
                           engine="python", quoting=csv.QUOTE_NONE,
                           on_bad_lines="skip")

def tidy_aug_dec(csv_path: Path) -> pd.DataFrame:
    df = read_fast_then_fallback(csv_path)

    # normalise column names
    df.columns = [unidecode.unidecode(c).lower().strip('" ') for c in df.columns]

    # map to English headers
    df = df.rename(columns={k: HEADER_MAP.get(k, k) for k in df.columns})

    # drop duplicate ESP timestamp if present
    df = df.drop(columns=["fecha_esp32"], errors="ignore")

    # ensure datetime column exists
    if "datetime" not in df.columns:
        raise KeyError(f"'fecha_servidor' column not found; headers={df.columns.tolist()}")

    # parse datetime (day-first)
    df["datetime"] = pd.to_datetime(
        df["datetime"].astype(str).str.strip('" '),
        dayfirst=True,
        utc=True,
        errors="coerce"
    )

    # sort & numeric coercion
    df = df.dropna(subset=["datetime"]).sort_values("datetime")
    for col in df.columns.difference(["datetime", "mac", "weekday"]):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

print("→ Cleaning 2agosto – dic 2021.csv …")
aug_dec = tidy_aug_dec(RAW)
print(f"Parsed datetimes: {aug_dec['datetime'].notna().sum():,} rows")

aug_dec.to_parquet(OUT, index=False)
print(f"✓ wrote {OUT.name}  –  {len(aug_dec):,} rows  "
      f"|  {round(OUT.stat().st_size / 1024**2, 1)} MB")


→ Cleaning 2agosto – dic 2021.csv …
Parsed datetimes: 1,520,697 rows
✓ wrote workstation_2021_aug_dec_tidy.parquet  –  1,520,697 rows  |  9.0 MB


In [16]:
# ───── Clean World Bank Population CSV  →  long tidy Parquet ─────

RAW   = Path(RAW_DIR / "World_bank_population.csv")
OUT   = Path(CLEAN_DIR / "world_pop_long.parquet")


def tidy_world_bank_pop(csv_path: Path) -> pd.DataFrame:
    """
    Converts the wide World Bank format (columns = years) into
      country | iso3 | year | population
    Only keeps 1960-2024, drops missing values & aggregate regions.
    """
    df = pd.read_csv(csv_path, low_memory=False)

    # Typical World Bank headers
    id_cols = ["Country Name", "Country Code"]
    year_cols = [c for c in df.columns if c.isdigit()]

    # Wide → long
    df_long = (
        df.melt(id_vars=id_cols,
                value_vars=year_cols,
                var_name="year",
                value_name="population")
          .rename(columns={
              "Country Name": "country",
              "Country Code": "iso3"
          })
    )

    # keep only 1960-2024, numeric population
    df_long["year"] = df_long["year"].astype(int)
    df_long["population"] = pd.to_numeric(df_long["population"], errors="coerce")

    df_long = df_long.query("1960 <= year <= 2024 and population.notna()")

    # drop aggregate rows (World, High income, etc.)
    aggregates = ["World", "High income", "OECD members", "Euro area"]
    df_long = df_long[~df_long["country"].isin(aggregates)]

    return df_long.reset_index(drop=True)

print("→ Cleaning World Bank population …")
pop_long = tidy_world_bank_pop(RAW)
pop_long.to_parquet(OUT, index=False)

print(f"✓ wrote {OUT.name}  –  {len(pop_long):,} rows  "
      f"|  {pop_long['iso3'].nunique()} countries")
print(pop_long.head())


→ Cleaning World Bank population …
✓ wrote world_pop_long.parquet  –  16,935 rows  |  261 countries
                       country iso3  year   population
0                        Aruba  ABW  1960      54922.0
1  Africa Eastern and Southern  AFE  1960  130075728.0
2                  Afghanistan  AFG  1960    9035043.0
3   Africa Western and Central  AFW  1960   97630925.0
4                       Angola  AGO  1960    5231654.0


In [17]:
# ───── Clean global DC counts  →  dc_count_tidy.parquet ─────

RAW = Path(RAW_DIR / "data-centres-worldwide-general-dataset.csv")
OUT = Path(CLEAN_DIR / "global_dc_counts_tidy.parquet")

def tidy_global_dc(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    tidy = (
        df[["Region", "Country", "Total Number of data centre"]]   # exact header
          .rename(columns={
              "Region": "region",
              "Country": "country",
              "Total Number of data centre": "dc_count"
          })
          .assign(dc_count=lambda x: pd.to_numeric(x["dc_count"], errors="coerce"))
          .dropna(subset=["dc_count"])
          .reset_index(drop=True)
    )
    return tidy
    

print("→ Cleaning global data-centre counts …")
dc_counts = tidy_global_dc(RAW)
dc_counts.to_parquet(OUT, index=False)

print(f"✓ wrote {OUT.name}  –  {len(dc_counts):,} rows   "
      f"|  {dc_counts['country'].nunique()} countries")
print(dc_counts.head())


→ Cleaning global data-centre counts …
✓ wrote global_dc_counts_tidy.parquet  –  136 rows   |  135 countries
     region              country  dc_count
0  Americas   Dominican Republic         1
1  Americas            Guatemala         1
2  Americas              Bolivia         2
3  Americas            Venezuela         2
4  Americas  Trinidad and Tobago         3


In [9]:
usa_dc = dc_counts[dc_counts['country'].str.contains('United States|USA', case=False, na=False)]
print("USA Data Center count:")
print(usa_dc)

USA Data Center count:
      region                   country  dc_count
16  Americas  United States of America      2052


In [10]:
usa_pop = pop_long[pop_long['country'].str.contains('United States', case=False, na=False)]
print("USA Population Data:")
print(usa_pop)

USA Population Data:
             country iso3  year   population
246    United States  USA  1960  180671000.0
506    United States  USA  1961  183691000.0
766    United States  USA  1962  186538000.0
1026   United States  USA  1963  189242000.0
1286   United States  USA  1964  191889000.0
...              ...  ...   ...          ...
15877  United States  USA  2020  331577720.0
16138  United States  USA  2021  332099760.0
16399  United States  USA  2022  334017321.0
16660  United States  USA  2023  336806231.0
16921  United States  USA  2024  340110988.0

[65 rows x 4 columns]


In [20]:
# ---------- country-name consistency check  -----------------

dc  = pd.read_parquet(CLEAN_DIR/ "global_dc_counts_tidy.parquet")
pop = (
    pd.read_parquet(CLEAN_DIR / "world_pop_long.parquet")
      .query("year == 2024")           # most recent pop slice
)

# ── null-safe normaliser → returns None on failure ──────────────────
def norm(name):
    try:
        if pd.isna(name):
            return None
    except TypeError:
        return None
    try:
        return unidecode.unidecode(str(name)).strip().title()
    except Exception:
        return None

dc["country_norm"]  = dc["country"].map(norm)
pop["country_norm"] = pop["country"].map(norm)

# ── fill NaNs with numeric 0  ───────────────────────────────────────
dc["country_norm"]  = dc["country_norm"].fillna(0)
pop["country_norm"] = pop["country_norm"].fillna(0)

# ---- set comparisons ----------------------------------------------
dc_set  = set(dc["country_norm"])
pop_set = set(pop["country_norm"])

# remove the numeric sentinel 0 before sorting
only_in_dc  = sorted(x for x in (dc_set  - pop_set) if x != 0)
only_in_pop = sorted(x for x in (pop_set - dc_set) if x != 0)

print("⮕ Countries only in DC counts:", only_in_dc, "\n")
print("⮕ Countries only in population:", only_in_pop, "\n")

⮕ Countries only in DC counts: ['Bahrainn', 'Cape Verde', 'Congo', 'Czech Republic', 'Egypt', 'Gambia', 'Iran (Islamic Republic Of)', 'Ivory Coast', 'Luxemburg', 'Macedonia', 'Moldavia', 'New Zeland', 'Polynesia', 'Sao Tome E Principe', 'Slovakia', 'South Korea', 'Turkey', 'United States Of America', 'Venezuela', 'Vietnam'] 

⮕ Countries only in population: ['Africa Eastern And Southern', 'Africa Western And Central', 'Albania', 'American Samoa', 'Andorra', 'Antigua And Barbuda', 'Arab World', 'Aruba', 'Bahamas, The', 'Bahrain', 'Barbados', 'Belize', 'Bermuda', 'Bhutan', 'British Virgin Islands', 'Brunei Darussalam', 'Burundi', 'Cabo Verde', 'Caribbean Small States', 'Cayman Islands', 'Central African Republic', 'Central Europe And The Baltics', 'Channel Islands', 'Comoros', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote D'Ivoire", 'Cuba', 'Curacao', 'Czechia', 'Dominica', 'Early-Demographic Dividend', 'East Asia & Pacific', 'East Asia & Pacific (Excluding High Income)', 'East Asia & Pacific